# P2 MobileViTv2 ImageNet + LI architecture control

**Scientific question:** What is the architecture contribution relative to P1-noSSL when initialization family, LI fusion, targets, loss, split, seed, and supervised protocol are fixed?
**Configuration:** `config/experiments/P2_mobilevitv2_imagenet_li_bestreg.yaml`
**Dataset:** 1,927 clean unique labeled images.
**Split:** 1,407 train / 289 validation / 231 sealed test; SHA256 `8927b822...3eae2f`.
**Checkpoint/initialization:** `timm/mobilevitv2_050.cvnets_in1k` ImageNet weights; no VICReg.
**Expected outputs:** external P2 best/epoch-60 checkpoints, history, validation artifacts, metadata, then the four-model frozen registry.

Run after notebooks 01–03 have completed. This notebook trains for all 60 epochs and never creates a test dataset or loader.


## 1. Deterministic environment setup

Set the CUDA deterministic workspace before importing PyTorch, then load only train/validation workflow functions.


In [ ]:
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
from pathlib import Path
import sys

REPO = Path.cwd().resolve()
while REPO != REPO.parent and not (REPO / "pyproject.toml").is_file():
    REPO = REPO.parent
if not (REPO / "pyproject.toml").is_file():
    raise RuntimeError("Open this notebook from inside the SoilNet repository")
sys.path.insert(0, str(REPO / "src"))

import json
import torch
from soilnet.final_sequence import verify_p2_fairness, write_frozen_model_registry
from soilnet.training import run_one_batch_preflight, train_experiment
from soilnet.utils import load_experiment_context, print_environment

CONFIG_PATH = REPO / "config/experiments/P2_mobilevitv2_imagenet_li_bestreg.yaml"


## 2. Locked hashes and fair-control protocol

Validate the exact split, seed, optimizer, transformations, LI semantics, BESTREG rule, and the intended architecture-only delta against P1-noSSL.


In [ ]:
fairness = verify_p2_fairness()
context = load_experiment_context(CONFIG_PATH)
assert context.split_sha256 == "8927b8223b8c4c234d264ad9ea62ac2df6161124a79787a2e71eeb5cd23eae2f"
assert context.config["timm_model_name"] == "mobilevitv2_050.cvnets_in1k"
assert context.config["initialization"] == "imagenet" and context.config["ssl_checkpoint"] is None
assert context.config["use_li"] is True
assert context.config["selection_formula"] == "(SM0_RMSE + SM20_RMSE) / 2"
print_environment(context)
print(fairness)


## 3. CUDA gate and new-output safety

The external P2 destination must be absent or contain only a valid rolling resume. A completed run is never overwritten.


In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("GPU_BLOCKED: P2 training requires CUDA")
metadata_path = context.run_dir / "run_metadata.json"
if metadata_path.is_file() and json.loads(metadata_path.read_text(encoding="utf-8")).get("training_completed") is True:
    raise RuntimeError("STOP: P2 output directory already contains a completed run")
print({"device": torch.cuda.get_device_name(0), "output_directory": str(context.run_dir), "test": "NOT_OPENED"})


## 4. Temporary-model preflight

Build a disposable ImageNet-initialized baseline and check one train/validation batch. No optimizer step, checkpoint, or research metric is produced.


In [ ]:
preflight = run_one_batch_preflight(context, device=torch.device("cuda"))
if preflight.get("status") != "PASS" or preflight.get("optimizer_step_performed") is not False or preflight.get("test_loader_instantiated") is not False:
    raise RuntimeError("P2_PREFLIGHT_FAILED")
print(preflight)


## 5. Automatic 60-epoch supervised run

Train all 60 epochs with Adam 1e-4, batch 32, weight decay 0, and no early stopping. Strictly lower mean validation regression RMSE replaces the best checkpoint.


In [ ]:
run_metadata = train_experiment(context, resume_if_available=True)
assert run_metadata["training_completed"] is True
assert run_metadata["epochs_completed"] == 60
assert run_metadata["test_evaluated"] == "NO"


## 6. Freeze all four models

Verify P0, both P1 ablations, and P2 checkpoint identities plus validation artifacts, then write the registry while the test firewall remains closed.


In [ ]:
registry = write_frozen_model_registry()
assert registry["model_count"] == 4
assert registry["TEST_OPENED"] == "NO" and registry["MODEL_SET_FROZEN"] == "YES"
print(json.dumps({"registry": "results/model_registry/frozen_model_registry.json", "models": [m["experiment_id"] for m in registry["models"]], "TEST_OPENED": "NO"}, indent=2))
